In [1]:
!pip install -q streamlit pyngrok langchain langchain-community langchain-huggingface sentence-transformers faiss-cpu pypdf pydantic transformers accelerate

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.3/10.3 MB 71.6 MB/s eta 0:00:00:00:01:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 61.5 MB/s eta 0:00:00:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.8/18.8 MB 83.0 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 42.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 573.0/573.0 kB 27.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 69.4/69.4 kB 3.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.4/11.4 MB 81.9 MB/s eta 0:00:00:00:010:01


In [2]:
%%writefile app.py
import os
import json
import re
import torch
import streamlit as st
from typing import List
from pydantic import BaseModel, Field

from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_huggingface import HuggingFaceEmbeddings, HuggingFacePipeline
from langchain_community.vectorstores import FAISS
from langchain_core.prompts import PromptTemplate
from transformers import AutoModelForCausalLM, AutoTokenizer, pipeline

st.set_page_config(page_title="AI Personal Tutor", page_icon="🎓", layout="wide")

st.markdown(
    """
    <script>
    // Watch for the settings/kebab dropdown and remove the "Print" item.
    const removePrint = () => {
        const items = document.querySelectorAll('[role="menuitem"], [data-testid*="stMenu"] li, [class*="menu"] li');
        items.forEach(el => {
            if (el.textContent && el.textContent.trim() === "Print") el.remove();
        });
    };
    const observer = new MutationObserver(removePrint);
    observer.observe(document.body, { childList: true, subtree: true });
    removePrint();
    </script>
    """,
    unsafe_allow_html=True,
)
st.title("🎓 AI Personal Tutor")

defaults = {
    "vector_store": None,
    "chat_history": [],
    "current_topic": None,
    "context_text": "",
    "tutorial": None,
    "quiz_history": [],
}
for key, value in defaults.items():
    if key not in st.session_state:
        st.session_state[key] = value

st.sidebar.header("🔑 Hugging Face Setup")
hf_token = st.sidebar.text_input("HF Access Token (Optional for gated models)", type="password")
model_repo = st.sidebar.selectbox(
    "Hugging Face Local Model",
    [
        "Qwen/Qwen2.5-1.5B-Instruct",
        "Qwen/Qwen2.5-7B-Instruct",
        "mistralai/Mistral-7B-Instruct-v0.3",
        "meta-llama/Meta-Llama-3-8B-Instruct"
    ]
)

@st.cache_resource
def load_local_llm(repo_id: str, token: str = None):
    tokenizer = AutoTokenizer.from_pretrained(repo_id, token=token if token else None)

    device_map = "auto" if torch.cuda.is_available() else "cpu"
    torch_dtype = torch.float16 if torch.cuda.is_available() else torch.float32

    model = AutoModelForCausalLM.from_pretrained(
        repo_id,
        torch_dtype=torch_dtype,
        device_map=device_map,
        token=token if token else None,
        low_cpu_mem_usage=True
    )

    pipe = pipeline(
        "text-generation",
        model=model,
        tokenizer=tokenizer,
        max_new_tokens=1024,
        temperature=0.2,
        do_sample=True,
        return_full_text=False
    )

    return HuggingFacePipeline(pipeline=pipe)
class QuizQuestion(BaseModel):
    question: str = Field(description="Multiple choice question text")
    options: List[str] = Field(description="List of 4 distinct options")
    correct_answer: str = Field(description="Exact correct answer string from options")
    explanation: str = Field(default="", description="Short reason why the answer is correct")

class ExplanationOutput(BaseModel):
    topic: str = Field(description="Requested topic title")
    explanation: str = Field(description="Detailed teaching explanation")
    example: str = Field(description="Practical real-world example")
    quiz: QuizQuestion = Field(description="Quiz question testing comprehension")

def extract_json(raw_str: str) -> dict:
    """Pull the first complete JSON object out of the model output.

    Uses raw_decode so trailing text (repeated JSON blocks, prose, or
    leftover markdown fences) is ignored instead of raising 'Extra data'.
    """
    cleaned = raw_str.strip()
    cleaned = re.sub(r"^```(?:json)?\s*", "", cleaned)
    cleaned = re.sub(r"\s*```$", "", cleaned)
    decoder = json.JSONDecoder()
    for i, ch in enumerate(cleaned):
        if ch == "{":
            try:
                obj, _ = decoder.raw_decode(cleaned[i:])
                if isinstance(obj, dict):
                    return obj
            except json.JSONDecodeError:
                continue
    raise ValueError("No JSON object found in model output.")

def resolve_correct_answer(options: List[str], correct: str):
    """Map the model's 'correct_answer' onto one of the options (handles 'B', 'B)', case, etc.)."""
    correct_norm = correct.strip().lower()
    for opt in options:
        if opt.strip().lower() == correct_norm:
            return opt
    letter = re.match(r"^\(?([a-d])[\)\.\:]?\s*$", correct_norm)
    if letter:
        idx = ord(letter.group(1)) - ord("a")
        if idx < len(options):
            return options[idx]
    for opt in options:
        if correct_norm and (correct_norm in opt.strip().lower() or opt.strip().lower() in correct_norm):
            return opt
    return None

def finalize_question(q: QuizQuestion) -> dict:
    """Validate a generated question and convert it to a plain dict for session state."""
    options = [o.strip() for o in q.options if o and o.strip()]
    if len(options) < 2:
        raise ValueError("Quiz question has fewer than 2 options.")
    correct = resolve_correct_answer(options, q.correct_answer)
    if correct is None:
        raise ValueError("Correct answer does not match any option.")
    return {
        "question": q.question.strip(),
        "options": options,
        "correct_answer": correct,
        "explanation": q.explanation.strip(),
        "selected": None, 
        "score": None,      
    }
@st.cache_resource
def get_embeddings():
    return HuggingFaceEmbeddings(
        model_name="sentence-transformers/all-MiniLM-L6-v2",
        encode_kwargs={"normalize_embeddings": True},
    )

def process_pdf(uploaded_file):
    temp_file_path = f"./temp_{uploaded_file.name}"
    with open(temp_file_path, "wb") as f:
        f.write(uploaded_file.getbuffer())

    loader = PyPDFLoader(temp_file_path)
    docs = loader.load()

    text_splitter = RecursiveCharacterTextSplitter(chunk_size=600, chunk_overlap=100)
    splits = text_splitter.split_documents(docs)

    embeddings = get_embeddings()
    vectorstore = FAISS.from_documents(splits, embeddings)

    os.remove(temp_file_path)
    return vectorstore

st.sidebar.header("📚 Study Materials")
uploaded_pdf = st.sidebar.file_uploader("Upload Lecture Notes (PDF)", type=["pdf"])

if uploaded_pdf and st.sidebar.button("Index Document"):
    with st.spinner("Processing PDF and indexing into Vector Store..."):
        st.session_state.vector_store = process_pdf(uploaded_pdf)
        st.sidebar.success("PDF indexed successfully!")

# Fixed at the maximum strictness (0.8 cosine similarity). Only chunks that closely
# match the topic are used. Lower this value if valid topics get rejected.
MIN_SIMILARITY = 0.8

def retrieve_relevant(vector_store, topic: str, k: int = 3, min_similarity: float = 0.8):
    """Return only chunks that are actually similar to the topic (not just the 'least bad' ones)."""
    results = vector_store.similarity_search_with_score(topic, k=k)
    relevant = []
    for doc, l2_dist in results:
        # Embeddings are normalized, so cosine = 1 - (L2^2 / 2)
        cosine = 1 - (l2_dist ** 2) / 2
        if cosine >= min_similarity:
            relevant.append(doc)
    return relevant

TUTORIAL_SCHEMA = """Output must be raw JSON with this exact key structure:
{
  "topic": "Topic Name",
  "explanation": "Clear explanation",
  "example": "Real world example",
  "quiz": {
    "question": "Question text",
    "options": ["Option A", "Option B", "Option C", "Option D"],
    "correct_answer": "Exact option text",
    "explanation": "One or two sentences on why this answer is correct"
  }
}"""

QUESTION_SCHEMA = """Output must be raw JSON with this exact key structure:
{
  "question": "Question text",
  "options": ["Option A", "Option B", "Option C", "Option D"],
  "correct_answer": "Exact option text",
  "explanation": "One or two sentences on why this answer is correct"
}"""

MAX_RETRIES = 3

class NotInDocumentError(Exception):
    pass

def generate_tutorial(llm, context_text: str, topic: str):
    prompt = PromptTemplate(
        template="""You are an expert AI Personal Tutor. Teach the student using ONLY the provided context.
Do NOT use outside knowledge. If the context does not contain information about the topic,
output exactly: {{"error": "not_in_document"}}
Otherwise, output MUST be formatted strictly as a single raw JSON object. Do not wrap in markdown syntax.

{schema_instructions}

Context:
{context}

Topic: {topic}

JSON Output:""",
        input_variables=["context", "topic"],
        partial_variables={"schema_instructions": TUTORIAL_SCHEMA}
    )
    chain = prompt | llm

    last_error = None
    for _ in range(MAX_RETRIES):
        try:
            raw = chain.invoke({"context": context_text, "topic": topic})
            data = extract_json(raw)
            if data.get("error") == "not_in_document":
                raise NotInDocumentError("That topic isn't covered in your uploaded document.")
            parsed = ExplanationOutput(**data)
            return parsed, finalize_question(parsed.quiz)
        except NotInDocumentError:
            raise
        except Exception as e:
            last_error = e
    raise RuntimeError(f"Could not generate a valid tutorial after {MAX_RETRIES} attempts: {last_error}")

def generate_more_question(llm, context_text: str, topic: str, previous_questions: List[str]):
    prompt = PromptTemplate(
        template="""You are an expert AI Personal Tutor. Write ONE new multiple choice question about the topic,
using ONLY the provided context. Do NOT use outside knowledge. The question must be different from the previous questions listed below.
If the context does not contain information about the topic, output exactly: {{"error": "not_in_document"}}
Otherwise, output MUST be formatted strictly as a single raw JSON object. Do not wrap in markdown syntax.

{schema_instructions}

Context:
{context}

Topic: {topic}

Previous questions (do NOT repeat these):
{previous}

JSON Output:""",
        input_variables=["context", "topic", "previous"],
        partial_variables={"schema_instructions": QUESTION_SCHEMA}
    )
    chain = prompt | llm

    previous_text = "\n".join(f"- {q}" for q in previous_questions) or "None"
    last_error = None
    for _ in range(MAX_RETRIES):
        try:
            raw = chain.invoke({"context": context_text, "topic": topic, "previous": previous_text})
            data = extract_json(raw)
            if data.get("error") == "not_in_document":
                raise NotInDocumentError("That topic isn't covered in your uploaded document.")
            q = finalize_question(QuizQuestion(**data))
            if q["question"].lower() in [p.lower() for p in previous_questions]:
                raise ValueError("Model repeated a previous question.")
            return q
        except NotInDocumentError:
            raise
        except Exception as e:
            last_error = e
    raise RuntimeError(f"Could not generate a new question after {MAX_RETRIES} attempts: {last_error}")

def sync_memory_log():
    """Keep the memory-tab entry for the current topic up to date."""
    if st.session_state.chat_history and st.session_state.chat_history[-1].get("active"):
        st.session_state.chat_history[-1]["questions"] = st.session_state.quiz_history

tab_learn, tab_memory = st.tabs(["📖 Learn & Practice", "💬 Tutor Memory Log"])

with tab_learn:
    if not st.session_state.vector_store:
        st.info("Upload a PDF document in the sidebar to initialize the AI Personal Tutor.")
    else:
        with st.spinner(f"Loading local Hugging Face model `{model_repo}` into memory..."):
            llm = load_local_llm(model_repo, hf_token)
        st.subheader("1. Learn a Topic")
        user_topic = st.text_input("Enter a concept or topic from your notes:", placeholder="e.g., Backpropagation")

        if st.button("Generate Tutorial"):
            if user_topic:
                with st.spinner("Searching document & executing local inference..."):
                    retrieved_docs = retrieve_relevant(
                        st.session_state.vector_store, user_topic, k=3, min_similarity=MIN_SIMILARITY
                    )
                    context_text = "\n\n".join([doc.page_content for doc in retrieved_docs])

                    try:
                        if not retrieved_docs:
                            raise NotInDocumentError(
                                f"I couldn't find anything about '{user_topic}' in your uploaded document. "
                                "Try a topic that appears in your notes."
                            )
                        parsed, first_question = generate_tutorial(llm, context_text, user_topic)
                        for entry in st.session_state.chat_history:
                            entry["active"] = False

                        st.session_state.current_topic = user_topic
                        st.session_state.context_text = context_text
                        st.session_state.tutorial = {
                            "topic": parsed.topic,
                            "explanation": parsed.explanation,
                            "example": parsed.example,
                        }
                        st.session_state.quiz_history = [first_question]
                        st.session_state.chat_history.append({
                            "topic": user_topic,
                            "tutorial": st.session_state.tutorial,
                            "questions": st.session_state.quiz_history,
                            "active": True,
                        })
                    except NotInDocumentError as e:
                        st.warning(str(e))
                    except Exception as e:
                        st.error(f"Inference/Parsing Error: {e}")
            else:
                st.warning("Please enter a topic first.")

        if st.session_state.tutorial:
            tut = st.session_state.tutorial
            st.markdown(f"### 📘 Concept: {tut['topic']}")
            st.write(tut["explanation"])
            st.info(f"**Real-world Example:** {tut['example']}")

        if st.session_state.quiz_history:
            st.divider()
            st.subheader("2. Check Your Understanding")

            quiz_history = st.session_state.quiz_history

            for i, q in enumerate(quiz_history):
                st.markdown(f"**Question {i + 1}:** {q['question']}")

                if q["score"] is None:

                    selected = st.radio(
                        "Select an answer:",
                        q["options"],
                        index=None,
                        key=f"radio_{st.session_state.current_topic}_{i}",
                    )
                    if st.button("Submit Answer", key=f"submit_{st.session_state.current_topic}_{i}"):
                        if selected is None:
                            st.warning("Please select an answer before submitting.")
                        else:
                            q["selected"] = selected
                            q["score"] = 1 if selected == q["correct_answer"] else 0
                            sync_memory_log()
                            st.rerun()
                else:
                    st.write(f"Your answer: {q['selected']}")
                    st.markdown(f"##### 📊 Score: {q['score']} / 1")
                    if q["score"] == 1:
                        st.success(f"✅ Correct! {q['explanation']}")
                    else:
                        st.error(
                            f"❌ Incorrect. The correct answer is: **{q['correct_answer']}**. {q['explanation']}"
                        )
                st.write("")

            answered = [q for q in quiz_history if q["score"] is not None]
            if answered:
                total = sum(q["score"] for q in answered)
                st.markdown(f"### 🏆 Total Score: {total} / {len(answered)}")

            if quiz_history[0]["score"] is not None:
                if st.button("➕ Generate More Questions"):
                    with st.spinner("Generating a new question..."):
                        try:
                            new_q = generate_more_question(
                                llm,
                                st.session_state.context_text,
                                st.session_state.current_topic,
                                [q["question"] for q in quiz_history],
                            )
                            quiz_history.append(new_q)
                            sync_memory_log()
                            st.rerun()
                        except NotInDocumentError as e:
                            st.warning(str(e))
                        except Exception as e:
                            st.error(f"Question generation error: {e}")

with tab_memory:
    st.subheader("Session Memory & Conversation Log")
    if not st.session_state.chat_history:
        st.write("No topics covered in this session yet.")
    else:
        for idx, entry in enumerate(st.session_state.chat_history):
            qs = entry.get("questions", [])
            answered = [q for q in qs if q["score"] is not None]
            total = sum(q["score"] for q in answered)
            label = f"Topic {idx + 1}: {entry['topic']}  —  Score {total} / {len(answered)}"
            with st.expander(label):
                st.json({"tutorial": entry["tutorial"], "questions": qs})

Writing app.py


In [3]:
import subprocess
import time
from pyngrok import ngrok
from kaggle_secrets import UserSecretsClient

!pkill -f streamlit || true
!pkill -f ngrok || true

NGROK_TOKEN = UserSecretsClient().get_secret("NGROK_TOKEN")
ngrok.set_auth_token(NGROK_TOKEN)

process = subprocess.Popen(
    ["streamlit", "run", "app.py", "--server.port", "8501", "--server.address", "0.0.0.0"]
)
time.sleep(5)

public_url = ngrok.connect(8501)
print(f"🚀 AI Personal Tutor live at: {public_url}")

2026-10-09 14:08:37.858 Uvicorn server started on 0.0.0.0:8501



  You can now view your Streamlit app in your browser.

  Local URL: http://localhost:8501
  Network URL: http://172.19.2.2:8501
  External URL: http://34.59.161.98:8501

🚀 AI Personal Tutor live at: NgrokTunnel: "https://vitamins-granola-detection.ngrok-free.dev" -> "http://localhost:8501"
